# ViTucano-1b5 LoRA fine-tuning (Kaggle)
GPU on, Internet on, Secret `HF_API_KEY`. Run cells 1-3 once, then cell 4 with `SMOKE=True`, then `SMOKE=False`.
For the real run use **Save Version -> Save & Run All**. Config lives in `vlm/src/config/config_vitucano.yml`.

In [ ]:
SMOKE = True       # True: 20 train / 10 test images (~3 min)
TRAIN_IMAGES = 2000   # real run size (None = all 5000, ~5-8h)
PUSH = True           # push adapter to the HF Hub (real run only)
BRANCH = "kaggle-vitucano"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ["HF_API_KEY"] = sec.get_secret("HF_API_KEY")
try: os.environ["WANDB_API_KEY"] = sec.get_secret("WANDB_API_KEY")
except Exception: pass
os.environ.update(USE_TF="0", USE_FLAX="0", CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1")

In [ ]:
%cd /kaggle/working
!git clone -q -b {BRANCH} https://github.com/LukaOliver-S/transformer-caption-ptbr.git 2>/dev/null || true
%cd /kaggle/working/transformer-caption-ptbr/vlm
!pip install -q transformers==4.45.2 peft==0.10.0 accelerate==1.0.1 datasets==3.0.2 evaluate==0.4.3     rouge_score bert_score nltk wandb python-dotenv huggingface_hub==0.26.1 open_clip_torch==2.22 openai==1.59.8 pyyaml tqdm
!pip install -q "protobuf>=5.29,<6"
with open(".env", "w") as f:
    f.write(f'HF_API_KEY="{os.environ["HF_API_KEY"]}"
')
    if "WANDB_API_KEY" in os.environ: f.write(f'WANDB_API_KEY="{os.environ["WANDB_API_KEY"]}"
')

In [ ]:
import shutil
%cd /kaggle/working/transformer-caption-ptbr/vlm/src
shutil.rmtree("../models", ignore_errors=True)   # clean run. To RESUME an interrupted run, comment this out.
flags = "--train-images 20 --test-images 10 --monitoring-steps 10 --no-push" if SMOKE else         (f"--train-images {TRAIN_IMAGES} " if TRAIN_IMAGES else "") + ("" if PUSH else "--no-push")
!python -u train_vitucano.py {flags}

In [ ]:
import pandas as pd
base = "../models/vitucano-1b-ft/flickr30k_pt/results/flickr30k_pt"
print(pd.read_csv(f"{base}/sample_eval_metrics.csv").T.to_string())
print(pd.read_json(f"{base}/predictions.json", lines=True)[["prediction_text", "label_text"]].head(6).to_string())

In [ ]:
!cd /kaggle/working/transformer-caption-ptbr/vlm && zip -rq /kaggle/working/vitucano_results.zip models -x "*/optimizer.pt" "*/rng_state.pth" "*/scheduler.pt"
!ls -lh /kaggle/working/vitucano_results.zip